##### 코랩을 사용할 경우

In [7]:
try:
    # Google Drive를 Colab(/content/drive)에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [8]:
%%capture 
# %%capture
# - 셀의 출력을 숨기는 셀 매직 명령어(%%)
# - 셀의 맨 첫 줄에 작성해야 하며, 셀의 출력이 많을 때 유용
# %run
# - 다른 노트북 파일을 실행하기 위한 라인 매직 명령어(%)
# - get_ipython().run_line_magic()은 %run 명령어를 Python 코드로 표현한 것
# - %%capture 안에서는 %run 매직을 직접 쓸 수 없는 경우가 있어서 Python 함수 형태로 호출
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")
get_ipython().run_line_magic("run", "02_model_definition.ipynb")

##### 임포트

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

##### Device 설정

In [10]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 옵티마이저 생성

In [11]:
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()

# Adam 옵티마이저(Optimizer) 생성
# - lr=0.001: 학습률 (모든 파라미터(가중치, 편향)를 얼마나 크게 변경할지)
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 학습하기

In [12]:
# 학습 에포크와 배치 크기 설정
epochs = 10
batch_size = 16

# 에포크 반복
for epoch in range(epochs):
    # 훈련 하기 -----------------------------------------------------
    # 학습 모드로 설정: 파라미터 업데이트 및 드롭아웃 활성화
    model.train()

    # 훈련 손실과 정확도 누적 변수 초기화
    batch_loss_sum   = 0
    train_correct = 0

    # 미니배치 학습
    for i in range(0, len(train_x_tensor), batch_size):
        batch_x_tensor = train_x_tensor[i:i+batch_size]
        batch_y_tensor = train_y_tensor[i:i+batch_size]

        # 1. 기울기(그래디언트) 초기화
        optimizer.zero_grad()

        # 2. 모델 출력 얻기
        # 내부적으로 nn.Module.__call__() → forward() 호출
        batch_train_outputs = model(batch_x_tensor)

        # 3. 손실 계산
        loss = loss_fn(batch_train_outputs, batch_y_tensor)

        # 4. 역전파: 기울기(그래디언트) 계산
        loss.backward()

        # 5. 파라미터 업데이트
        optimizer.step()

        # 각 배치 손실의 합산
        batch_loss_sum += loss.item()

        # 각 배치에서 맞춘 개수 합산: sum(): True(=1) 개수 합산
        train_correct += (batch_train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 훈련 평균 손실: 각 배치 손실의 합산 / 배치 개수
    epoch_loss = batch_loss_sum / (len(train_x_tensor) / batch_size)

    # 훈련 정확도: 맞춘 개수 / 전체 개수 * 100
    epoch_acc = train_correct / len(train_x_tensor) * 100

    # 출력하기 -----------------------------------------------------
    print(f"Epoch [{epoch+1:2d}/{epochs}] "
          f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:.2f}%")

Epoch [ 1/10] 훈련 손실: 1.0979  훈련 정확도: 40.85%
Epoch [ 2/10] 훈련 손실: 1.0180  훈련 정확도: 62.68%
Epoch [ 3/10] 훈련 손실: 0.9557  훈련 정확도: 72.54%
Epoch [ 4/10] 훈련 손실: 0.8573  훈련 정확도: 82.39%
Epoch [ 5/10] 훈련 손실: 0.7536  훈련 정확도: 88.73%
Epoch [ 6/10] 훈련 손실: 0.6411  훈련 정확도: 92.96%
Epoch [ 7/10] 훈련 손실: 0.5285  훈련 정확도: 95.07%
Epoch [ 8/10] 훈련 손실: 0.4169  훈련 정확도: 95.07%
Epoch [ 9/10] 훈련 손실: 0.3138  훈련 정확도: 96.48%
Epoch [10/10] 훈련 손실: 0.2693  훈련 정확도: 95.77%
